## Breast Cancer Wisconsin dataset

このデータセットは、乳房腫瘤の **FNA（fine needle aspirate, 穿刺吸引細胞診）画像** から抽出した細胞核の特徴量を使って、腫瘍が悪性か良性かを判定するためのもの

- サンプル数: 569
- 説明変数: 30 個
- 目的変数: `target`
- クラス:
  - `0`: malignant（悪性）
  - `1`: benign（良性）

各細胞核について、次の 10 種類の特徴が計算されている

- `radius`: 中心から輪郭上の点までの距離の平均
- `texture`: 画像の濃淡値の標準偏差
- `perimeter`: 周囲長
- `area`: 面積
- `smoothness`: 半径の局所的な変動
- `compactness`: $\frac{\mathrm{perimeter}^2}{\mathrm{area}} - 1$
- `concavity`: 輪郭のへこみの強さ
- `concave points`: 輪郭のへこんだ点の数
- `symmetry`: 対称性
- `fractal dimension`: 輪郭の複雑さを表す指標

上の 10 種類の基本特徴について、それぞれ次の 3 種類が計算されているため、全部で $10 \times 3 = 30$ 変数ある

- `mean`: 平均値
- `error`: 標準誤差 (`se` と表現されることもあります)
- `worst`: 最も大きい側の値の代表

## 使用するライブラリの読み込み

データ処理、可視化、データ分割、評価指標、Random Forest モデルに必要なライブラリを読み込んでいる。

ここで特に重要なのは次の 3 つ

- `load_breast_cancer`: scikit-learn に内蔵されているデータセットを読み込む
- `train_test_split`: 学習用データとテスト用データに分割する
- `RandomForestClassifier`: Random Forest による分類モデルを作る


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.ensemble import RandomForestClassifier

## データセットの読み込みと確認

このセルでは、Breast Cancer Wisconsin dataset を読み込み、説明変数 `X` と目的変数 `y` を作っている

- `X`: 30 個の数値特徴量
- `y`: 良性・悪性を表すクラスラベル

さらに、データの行列サイズとクラス分布を表示して、

- サンプル数がいくつか
- クラスの偏りが大きすぎないか

を確認しています。

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())

display(X.head())
display(y.head())
print(data.DESCR)

## 学習データとテストデータへの分割、および Random Forest の学習

このセルでは、データを訓練用と評価用に分割し、その後 Random Forest を学習している。

処理の流れは

- `train_test_split` でデータを訓練用 80%、テスト用 20% に分割する
- `stratify=y` によって、良性・悪性の比率が train と test で大きく崩れないようにする
- `RandomForestClassifier` を定義する
- `fit` によって訓練データでモデルを学習する

ここでは `oob_score=True` を指定しているため、学習後に OOB score と OOB error を確認できる
OOB error は、学習に使われなかったサンプルを使って内部的に見積もる誤差


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf_bc = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

rf_bc.fit(X_train, y_train)

## テストデータでの予測と性能評価

このセルでは、学習済みモデルを使ってテストデータを予測し、分類性能を数値で評価している

主に見ているのは次の指標

- `Accuracy`: 全体として何割正しく分類できたか
- `ROC-AUC`: 良性と悪性をどれだけうまく分離できるか
- `OOB score`, `OOB error`: Random Forest の内部評価
- `classification_report`: precision, recall, f1-score の一覧
- `confusion_matrix`: 実際のクラスと予測クラスの対応表

このセルによって、モデルがどの程度うまく良性・悪性を判別できているかを確認する


In [ ]:
y_pred = rf_bc.predict(X_test)
y_proba = rf_bc.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("OOB score:", rf_bc.oob_score_)
print("OOB error:", 1 - rf_bc.oob_score_)
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

## ROC 曲線と Precision-Recall 曲線の可視化

このセルでは、分類性能を曲線として可視化する

- `ROC Curve`: しきい値を動かしたときの偽陽性率と真陽性率の関係を見る
- `Precision-Recall Curve`: precision と recall のトレードオフを見る

モデルの性能を単なる指標だけでなく、より詳細に評価するためのセル

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Breast Cancer Wisconsin - ROC Curve")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.title("Breast Cancer Wisconsin - Precision-Recall Curve")
plt.show()